# Domain Adaptive Pre-Training (DAPT) with NeMo AutoModel

**Domain Adaptive Pre-Training (DAPT)** continues pre-training a general-purpose
foundation model on a curated, domain-specific corpus. Instead of training from
scratch, DAPT leverages the broad capabilities already encoded in the foundation
model and specializes them for a target domain such as chip design, biomedicine,
finance, or law.

This approach was demonstrated at scale in the
[ChipNeMo paper (Arxiv 2311.00176)](https://arxiv.org/abs/2311.00176), where
continued pre-training on chip-design data significantly improved performance on
engineering assistant tasks while retaining general language ability.

This tutorial walks through end-to-end DAPT using **NeMo AutoModel** and a
YAML-driven configuration.

| Step | Description |
|------|-------------|
| **Step 0** | Environment setup and verification |
| **Step 1** | Data preparation (JSONL to bin/idx) |
| **Step 2** | Configure training via YAML |
| **Step 3** | Launch training with the `automodel` CLI |
| **Step 4** | Evaluate the domain-adapted model |

## Prerequisites

Before starting, make sure you have:

- **NeMo AutoModel** installed (see the [AutoModel README](https://github.com/NVIDIA-NeMo/Automodel) for setup instructions).
- **GPU(s)**: Tested on 2x H100 80 GB. Scale up or down by adjusting `--nproc-per-node` and distributed settings in the config.
- **Hugging Face token** for gated model access (Llama). Run `huggingface-cli login` or set the `HF_TOKEN` environment variable.
- **Domain data** curated and ready. See [NeMo Curator](https://github.com/NVIDIA/NeMo-Curator) for large-scale data curation pipelines.

## Step 0 -- Environment Setup

The recommended way to run NeMo AutoModel is inside the NeMo AutoModel container.

```bash
# Launch the NeMo AutoModel container (replace the tag with the latest available)
docker run --gpus all -it --rm \
    --shm-size=16g \
    --ulimit memlock=-1 \
    --ulimit stack=67108864 \
    -v $(pwd):/workspace \
    nvcr.io/nvidia/nemo-automodel:26.02.00 \
    bash

# Inside the container, launch Jupyter (optional)
jupyter lab --ip 0.0.0.0 --port 8888 --no-browser --allow-root
```

In [ ]:
# Step 0: Verify the environment
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    mem_gb = props.total_memory / 1024**3
    print(f"GPU: {props.name}")
    print(f"Memory: {mem_gb:.1f} GB")

## Step 1 -- Data Preparation

NeMo AutoModel uses the **Megatron dataset format** (binary `.bin` + index
`.idx` files) for efficient, random-access pre-training data loading. This
format avoids repeated tokenization at runtime and supports large-scale
data blending.

The high-level workflow is:

1. Collect domain text (code, papers, documentation, etc.) into **JSONL**
   format with a `"text"` field per line.
2. Optionally curate and deduplicate with
   [NeMo Curator](https://github.com/NVIDIA/NeMo-Curator).
3. Convert the JSONL to `.bin`/`.idx` using the preprocessing script shown
   below.

In [ ]:
# Step 1: Preprocess domain data from JSONL to Megatron bin/idx format.
# NeMo provides a preprocessing script for this conversion.
# Replace paths with your actual data locations.
#
# python scripts/nlp_language_modeling/preprocess_data_for_megatron.py \
#     --input /path/to/domain_data.jsonl \
#     --json-keys text \
#     --tokenizer-library huggingface \
#     --tokenizer-type meta-llama/Llama-3.2-1B \
#     --append-eod \
#     --output-prefix /path/to/output/domain_data
#
# This produces two files:
#   domain_data_text_document.bin  -- binary token data
#   domain_data_text_document.idx  -- index for fast random access
print("After preprocessing, you should have .bin and .idx files.")
print("See the NeMo documentation for the full preprocessing script.")


In [ ]:
# Verify the preprocessed files exist.
import os

DATA_DIR = "/path/to/preprocessed_data"  # REPLACE THIS with your actual directory
prefix = "domain_data_text_document"

print("Expected files:")
for ext in [".bin", ".idx"]:
    fpath = os.path.join(DATA_DIR, prefix + ext)
    exists = os.path.isfile(fpath)
    size = os.path.getsize(fpath) / (1024 ** 2) if exists else 0
    status = f"{size:.1f} MB" if exists else "NOT FOUND"
    print(f"  {fpath}  [{status}]")

### Data Preprocessing Details

1. **Collect** domain data (source code, research papers, internal docs, etc.)
   into JSONL with one JSON object per line containing at least a `"text"` key.

   ```json
   {"text": "The FinFET transistor architecture provides ..."}
   {"text": "module counter(input clk, output reg [7:0] count); ..."}
   ```

2. **Curate** with [NeMo Curator](https://github.com/NVIDIA/NeMo-Curator):
   language filtering, deduplication, quality scoring, and PII removal.

3. **Convert** to bin/idx using the preprocessing command above. The tokenizer
   must match the model you will continue pre-training (Llama-3.2-1B here).

4. **Blend** multiple sources by listing them in the config with weights:

   ```yaml
   paths:
     - 0.7
     - /data/domain_corpus_text_document
     - 0.3
     - /data/general_corpus_text_document
   ```

## Step 2 -- Configure Training

NeMo AutoModel uses a single **YAML configuration file** to define every
aspect of training: model, dataset, distributed strategy, optimizer,
learning-rate schedule, checkpointing, and more.

For DAPT, we use the `TrainFinetuneRecipeForNextTokenPrediction` recipe --
the same recipe used for supervised fine-tuning, but pointed at raw
pre-training data instead of instruction-formatted data. The key differences
from SFT are:

- **Dataset**: `MegatronPretraining` (bin/idx token data) rather than a
  chat/instruction dataset.
- **Learning rate**: Lower (2e-5 vs. typical 5e-5 for SFT) to preserve base
  model capabilities while injecting domain knowledge.
- **Sequence packing**: Enabled to maximize GPU utilization on variable-length
  documents.

The config file is located at `dapt_config.yaml` in this directory.

In [ ]:
# Step 2: Display the training configuration.
from pathlib import Path

config_path = Path("dapt_config.yaml")
print(config_path.read_text())

### Key Configuration Choices

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `lr` | `2e-5` | Lower than SFT to avoid catastrophic forgetting of base capabilities |
| `packed_sequence_size` | `4096` | Pack variable-length documents into fixed-size sequences for higher throughput |
| `global_batch_size` | `128` | Large effective batch stabilizes pre-training loss |
| `model` | `from_pretrained` | Load HuggingFace-hosted weights as the starting point for continued training |
| `lr_decay_style` | `cosine` | Smooth decay matches typical pre-training schedules |
| `lr_warmup_steps` | `50` | Brief warmup avoids early instability when resuming training |
| `strategy` | `fsdp2` | Fully Sharded Data Parallelism for memory-efficient multi-GPU training |
| `weight_decay` | `0.01` | Light regularization consistent with Llama pre-training |
| `max_norm` | `1.0` | Gradient clipping prevents loss spikes |

Before launching, update the `paths` entries in `dapt_config.yaml` to
point to your preprocessed bin/idx files.

## Step 3 -- Launch Training

NeMo AutoModel provides the `automodel` CLI which reads the YAML config,
instantiates all components, and handles distributed launch via
`torchrun` under the hood.

Set `--nproc-per-node` to the number of GPUs you want to use.

In [ ]:
# Step 3: Launch DAPT with 2 GPUs (adjust --nproc-per-node to your setup).
!automodel tutorials/dapt/dapt_config.yaml --nproc-per-node 2

### Monitor Training

Training progress is logged to stdout by default. For richer experiment
tracking, add a Weights & Biases section to the YAML config:

```yaml
wandb:
  project: dapt-tutorial
  name: llama-3.2-1b-domain-dapt
```

Key metrics to watch during DAPT:

- **Training loss**: Should decrease steadily. A sudden spike may indicate a
  learning rate that is too high or a data quality issue.
- **Validation loss**: Computed every `val_every_steps` (100 steps in this
  config). Divergence from training loss signals overfitting.
- **Learning rate**: Verify the cosine schedule with warmup is applied
  correctly.
- **Gradient norm**: Should stay near or below `max_norm` (1.0). Persistent
  clipping may indicate instability.

## Step 4 -- Evaluate the Adapted Model

After DAPT completes, the checkpoint is saved in the directory specified by
`checkpoint.checkpoint_dir`. You can evaluate using:

1. **Perplexity** on a held-out domain validation set (quick sanity check).
2. **Benchmark evaluation** with
   [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness)
   to measure both domain and general capabilities.

In [ ]:
# Step 4: Quick perplexity check on domain text.
import glob
import math
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# --- Locate the latest checkpoint ---
CKPT_DIR = "checkpoints/"  # Update to your checkpoint path
ckpt_dirs = sorted(glob.glob(f"{CKPT_DIR}/epoch_*_step_*"))
if ckpt_dirs:
    latest_ckpt = ckpt_dirs[-1] + "/model/consolidated"
else:
    latest_ckpt = "meta-llama/Llama-3.2-1B"  # fallback for demo
print(f"Loading checkpoint: {latest_ckpt}")

# --- Load model and tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(latest_ckpt)
model = AutoModelForCausalLM.from_pretrained(
    latest_ckpt, torch_dtype=torch.bfloat16, device_map="auto"
)
model.eval()

# --- Sample domain text for evaluation ---
domain_samples = [
    "The FinFET transistor architecture provides superior electrostatic control",
    "Standard cell libraries are characterized by timing, power, and area metrics",
    "Clock tree synthesis minimizes skew while meeting insertion delay targets",
]

total_loss = 0.0
total_tokens = 0

with torch.no_grad():
    for text in domain_samples:
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        outputs = model(**inputs, labels=inputs["input_ids"])
        total_loss += outputs.loss.item() * inputs["input_ids"].numel()
        total_tokens += inputs["input_ids"].numel()

ppl = math.exp(total_loss / total_tokens)
print(f"Domain perplexity: {ppl:.2f}")


### Benchmark Evaluation

For a more thorough evaluation, use
[lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness)
with vLLM as the inference backend.

```bash
# Install lm-evaluation-harness
pip install lm-eval[vllm]

# Run evaluation on general benchmarks to verify no regression
lm_eval --model vllm \
    --model_args pretrained=checkpoints/consolidated,dtype=auto \
    --tasks hellaswag,mmlu \
    --batch_size auto
```

Compare the scores against the base model (before DAPT) to confirm that
domain knowledge was gained without significant regression on general tasks.
For a complete evaluation workflow, see the
[Evaluation tutorial](../evaluation/).

## Next Steps

After DAPT, the model has stronger domain knowledge but is not yet
instruction-tuned. The typical next steps are:

1. **Supervised Fine-Tuning (SFT)** or **Parameter-Efficient Fine-Tuning
   (PEFT / LoRA)** to teach the domain-adapted model to follow instructions.
   See the [SFT / PEFT tutorial](../sft-peft/).

2. **Alignment** via RLHF, DPO, or PPO to improve response quality and
   safety. See [NeMo-RL](https://github.com/NVIDIA/NeMo-RL).

3. **Evaluation** on domain-specific and general benchmarks to quantify
   improvements. See the [Evaluation tutorial](../evaluation/).

The full pipeline is:

```
Foundation Model --> DAPT --> SFT / PEFT --> Alignment --> Evaluation
```

## Resources

- [NeMo AutoModel GitHub](https://github.com/NVIDIA-NeMo/Automodel)
- [ChipNeMo: Domain-Adapted LLMs for Chip Design (Arxiv 2311.00176)](https://arxiv.org/abs/2311.00176)
- [NeMo Curator](https://github.com/NVIDIA/NeMo-Curator) -- large-scale data curation
- [NeMo-RL](https://github.com/NVIDIA/NeMo-RL) -- reinforcement learning from human feedback
- [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness) -- benchmark evaluation